# Esperimento Campo Morfogenetico EAR
## Test Quantitativo delle Proposizioni

**Data:** 11 Gennaio 2026  
**Framework:** Sistema Formale EAR v2.0

---

### Obiettivo

Testare quantitativamente le predizioni delle Proposizioni EAR:
- **Prop 3 (Soglia Critica):** La transizione è discreta, non graduale
- **Prop 4 (Scaling):** Pattern invariante per scala
- **Prop 6 (Inseparabilità):** Δ, ⇄, ⟳ co-variano

### Metodo

Confrontiamo pattern di Turing classici con pattern generati sotto l'influenza di un **kernel morfogenetico derivato dall'ontologia EAR**.

In [ ]:
# Import librerie
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, maximum_filter, sobel
from scipy.stats import pearsonr
from scipy.signal import correlate2d
import warnings
warnings.filterwarnings('ignore')

print("Librerie caricate.")

## 1. Derivazione del Kernel EAR

Il kernel è derivato dalla struttura ontologica:

- **Centro (3.0):** Campo ⧈ - massima potenzialità
- **Bordi cardinali (-0.5):** Distinzione Δ - separazione che definisce
- **Angoli (0.0):** Relazione ⇄ indiretta - non separano, connettono

La somma del kernel = 1.0 (conservazione informazionale, Prop 2)

In [ ]:
# Kernel EAR derivato dall'ontologia
KERNEL_EAR = np.array([
    [ 0.0, -0.5,  0.0],
    [-0.5,  3.0, -0.5],
    [ 0.0, -0.5,  0.0]
])

print("Kernel EAR:")
print(KERNEL_EAR)
print(f"\nSomma: {np.sum(KERNEL_EAR)} (deve essere > 0 per Prop 2)")

## 2. Simulatore Gray-Scott con Campo Morfogenetico

In [ ]:
def simulate_gray_scott(size=150, steps=3000, 
                        feed=0.055, kill=0.062,
                        Da=0.20, Db=0.10, dt=1.0,
                        morphic_kernel=None, 
                        field_strength=0.015,
                        application_freq=100,
                        seed=42):
    """
    Simula sistema Gray-Scott con opzionale campo morfogenetico EAR.
    """
    np.random.seed(seed)
    
    # Kernel Laplaciano per diffusione
    laplacian = np.array([
        [0.05, 0.2, 0.05],
        [0.2, -1.0, 0.2],
        [0.05, 0.2, 0.05]
    ])
    
    # Inizializzazione
    A = np.ones((size, size)) + np.random.normal(0, 0.05, (size, size))
    B = np.zeros((size, size)) + np.random.normal(0, 0.05, (size, size))
    
    # Seed centrale
    mid, r = size // 2, size // 10
    A[mid-r:mid+r, mid-r:mid+r] = 0.5
    B[mid-r:mid+r, mid-r:mid+r] = 0.25
    
    # Simulazione
    for step in range(steps):
        lap_A = convolve(A, laplacian, mode='wrap')
        lap_B = convolve(B, laplacian, mode='wrap')
        reaction = A * B * B
        
        A_new = A + (Da * lap_A - reaction + feed * (1 - A)) * dt
        B_new = B + (Db * lap_B + reaction - (kill + feed) * B) * dt
        
        # Campo morfogenetico
        if morphic_kernel is not None and step % application_freq == 0:
            influence = convolve(A_new, morphic_kernel, mode='wrap')
            B_new += field_strength * influence
        
        A = np.clip(A_new, 0, 1)
        B = np.clip(B_new, 0, 1)
    
    return A, B

print("Funzione di simulazione definita.")

## 3. Metriche Quantitative (mappate su Proposizioni)

In [ ]:
def calculate_metrics(image):
    """
    Metriche mappate su attributi EAR:
    - Centri forti → Nodi ⬡
    - Contrasto → Distinzione Δ
    - Lunghezza correlazione → Relazione ⇄
    - Entropia spettrale → Processo ⟳
    - Bilanciamento scala → Scaling (Prop 4)
    """
    metrics = {}
    
    # 1. Centri forti (⬡)
    local_max = maximum_filter(image, 7)
    centers = (image == local_max) & (image > np.mean(image))
    metrics['centri_forti'] = np.sum(centers)
    
    # 2. Contrasto (Δ)
    gradient = np.hypot(sobel(image, 0), sobel(image, 1))
    metrics['contrasto'] = np.std(gradient)
    
    # 3. Lunghezza correlazione (⇄)
    autocorr = correlate2d(image - np.mean(image), 
                           image - np.mean(image), mode='same')
    autocorr_norm = autocorr / (autocorr.max() + 1e-10)
    center = image.shape[0] // 2
    radial = autocorr_norm[center, center:]
    try:
        corr_length = np.where(radial < 0.5)[0][0]
    except:
        corr_length = len(radial)
    metrics['lunghezza_correlazione'] = corr_length
    
    # 4. Entropia spettrale (⟳)
    fft = np.fft.fft2(image)
    power = np.abs(fft) ** 2
    power = power / (power.sum() + 1e-10)
    power_flat = power.flatten()
    power_flat = power_flat[power_flat > 1e-10]
    metrics['entropia_spettrale'] = -np.sum(power_flat * np.log(power_flat))
    
    # 5. Bilanciamento scala (Prop 4)
    fft_shifted = np.fft.fftshift(fft)
    size = image.shape[0]
    center = size // 2
    y, x = np.ogrid[-center:size-center, -center:size-center]
    r = np.sqrt(x*x + y*y)
    
    low = np.sum(np.abs(fft_shifted[r < size/6]) ** 2)
    mid = np.sum(np.abs(fft_shifted[(r >= size/6) & (r < size/3)]) ** 2)
    high = np.sum(np.abs(fft_shifted[r >= size/3]) ** 2)
    total = low + mid + high + 1e-10
    
    metrics['bilanciamento_scala'] = (mid + high) / total
    
    return metrics

print("Funzione metriche definita.")

## 4. ESPERIMENTO 1: Confronto Pattern

Confrontiamo:
1. Turing classico (nessun campo)
2. Campo EAR moderato
3. Campo EAR forte (oltre soglia)

In [ ]:
print("Generazione pattern...")
print("  1/3 Turing classico...")
A_classic, _ = simulate_gray_scott(morphic_kernel=None)

print("  2/3 Campo EAR moderato...")
A_ear_mod, _ = simulate_gray_scott(morphic_kernel=KERNEL_EAR, field_strength=0.015)

print("  3/3 Campo EAR forte...")
A_ear_forte, _ = simulate_gray_scott(morphic_kernel=KERNEL_EAR, field_strength=0.05)

print("Fatto!")

In [ ]:
# Visualizzazione
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(A_classic, cmap='viridis')
axes[0].set_title('Turing Classico\n(Nessun campo)', fontsize=12)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(A_ear_mod, cmap='viridis')
axes[1].set_title('Campo EAR\n(strength=0.015)', fontsize=12)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(A_ear_forte, cmap='viridis')
axes[2].set_title('Campo EAR Forte\n(strength=0.05, oltre soglia)', fontsize=12)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle('Confronto Pattern: Turing vs Campo Morfogenetico EAR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Metriche comparative
m_classic = calculate_metrics(A_classic)
m_ear_mod = calculate_metrics(A_ear_mod)
m_ear_forte = calculate_metrics(A_ear_forte)

print("=" * 70)
print("METRICHE COMPARATIVE")
print("=" * 70)
print(f"{'Metrica':<25} {'Classico':>12} {'EAR mod':>12} {'EAR forte':>12}")
print("-" * 70)
for key in m_classic.keys():
    print(f"{key:<25} {m_classic[key]:>12.4f} {m_ear_mod[key]:>12.4f} {m_ear_forte[key]:>12.4f}")
print("=" * 70)

## 5. ESPERIMENTO 2: Test Soglia Critica (Prop 3)

**Predizione:** Esiste un valore di field_strength dove le metriche saltano discontinuamente.

In [ ]:
print("Scanning soglia critica...")
strengths = np.linspace(0, 0.06, 25)
centri_list = []
contrasto_list = []

for i, s in enumerate(strengths):
    A, _ = simulate_gray_scott(size=100, steps=2000, 
                                morphic_kernel=KERNEL_EAR, 
                                field_strength=s)
    m = calculate_metrics(A)
    centri_list.append(m['centri_forti'])
    contrasto_list.append(m['contrasto'])
    
    if (i+1) % 5 == 0:
        print(f"  {i+1}/25 completato")

print("Fatto!")

In [ ]:
# Visualizzazione soglia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(strengths, centri_list, 'b-o', linewidth=2, markersize=5)
ax1.axvline(x=0.05, color='r', linestyle='--', linewidth=2, label='Soglia critica')
ax1.set_xlabel('Field Strength', fontsize=12)
ax1.set_ylabel('Numero Centri (⬡)', fontsize=12)
ax1.set_title('Prop 3: Soglia Critica - Centri', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(strengths, contrasto_list, 'g-o', linewidth=2, markersize=5)
ax2.axvline(x=0.05, color='r', linestyle='--', linewidth=2, label='Soglia critica')
ax2.set_xlabel('Field Strength', fontsize=12)
ax2.set_ylabel('Contrasto (Δ)', fontsize=12)
ax2.set_title('Prop 3: Soglia Critica - Contrasto', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('TEST PROPOSIZIONE 3: Transizione Discreta', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Analisi discontinuità
d_centri = np.abs(np.diff(centri_list))
d_contrasto = np.abs(np.diff(contrasto_list))

sharpness = np.max(d_centri) / (np.mean(d_centri) + 1e-10)
print(f"\nSharpness ratio: {sharpness:.2f}")
print(f"Soglia rilevata a: {strengths[np.argmax(d_centri)]:.4f}")
print(f"→ {'TRANSIZIONE SHARP (Prop 3 confermata)' if sharpness > 3 else 'Transizione graduale'}")

## 6. ESPERIMENTO 3: Test Inseparabilità (Prop 6)

**Predizione:** I tre attributi Δ, ⇄, ⟳ sono correlati - non possono variare indipendentemente.

In [ ]:
print("Test inseparabilità attributi...")
feeds = np.linspace(0.04, 0.07, 12)
metrics_insep = []

for feed in feeds:
    A, _ = simulate_gray_scott(size=100, steps=2000,
                                morphic_kernel=KERNEL_EAR,
                                feed=feed)
    metrics_insep.append(calculate_metrics(A))

# Estrai serie
delta = [m['contrasto'] for m in metrics_insep]
rel = [m['lunghezza_correlazione'] for m in metrics_insep]
proc = [m['entropia_spettrale'] for m in metrics_insep]

# Correlazioni
corr_dr, p_dr = pearsonr(delta, rel)
corr_dp, p_dp = pearsonr(delta, proc)
corr_rp, p_rp = pearsonr(rel, proc)

print("\n" + "=" * 60)
print("TEST PROPOSIZIONE 6: Inseparabilità Attributi")
print("=" * 60)
print(f"Correlazione Δ ↔ ⇄: r = {corr_dr:.3f}, p = {p_dr:.4f}")
print(f"Correlazione Δ ↔ ⟳: r = {corr_dp:.3f}, p = {p_dp:.4f}")
print(f"Correlazione ⇄ ↔ ⟳: r = {corr_rp:.3f}, p = {p_rp:.4f}")

n_significant = sum(1 for r in [corr_dr, corr_dp, corr_rp] if abs(r) > 0.5)
print(f"\nCorrelazioni significative (|r| > 0.5): {n_significant}/3")
print(f"→ {'PROP 6 CONFERMATA: attributi co-variano' if n_significant >= 2 else 'Prop 6 non supportata'}")

## 7. ESPERIMENTO 4: Test Scaling (Prop 4)

**Predizione:** Il bilanciamento di scala è invariante attraverso le scale.

In [ ]:
print("Test scaling dimensionale...")
sizes = [75, 150, 300]
scaling_results = {}

for size in sizes:
    print(f"  Simulazione {size}x{size}...")
    A, _ = simulate_gray_scott(size=size, 
                                steps=int(3000 * size/150),
                                morphic_kernel=KERNEL_EAR)
    scaling_results[size] = calculate_metrics(A)

print("\n" + "=" * 60)
print("TEST PROPOSIZIONE 4: Scaling Dimensionale")
print("=" * 60)

for size in sizes:
    print(f"Scale {size}: bilanciamento = {scaling_results[size]['bilanciamento_scala']:.6f}")

bil_values = [scaling_results[s]['bilanciamento_scala'] for s in sizes]
variance = np.var(bil_values)
print(f"\nVarianza bilanciamento: {variance:.8f}")
print(f"→ {'PROP 4 CONFERMATA: invariante per scala' if variance < 0.0001 else 'Prop 4 parziale'}")

## 8. REPORT FINALE

In [ ]:
print("\n" + "=" * 70)
print("REPORT FINALE - TEST PROPOSIZIONI EAR")
print("=" * 70)

print("\n[PROP 3] SOGLIA CRITICA")
print(f"  Sharpness ratio: {sharpness:.2f}")
print(f"  Risultato: {'✓ CONFERMATA' if sharpness > 3 else '✗ NON CONFERMATA'}")

print("\n[PROP 4] SCALING DIMENSIONALE")
print(f"  Varianza bilanciamento: {variance:.8f}")
print(f"  Risultato: {'✓ CONFERMATA' if variance < 0.0001 else '~ PARZIALE'}")

print("\n[PROP 6] INSEPARABILITÀ ATTRIBUTI")
print(f"  Correlazioni significative: {n_significant}/3")
print(f"  Risultato: {'✓ CONFERMATA' if n_significant >= 2 else '✗ NON CONFERMATA'}")

print("\n" + "=" * 70)
print("CONCLUSIONE")
print("=" * 70)
total_confirmed = (sharpness > 3) + (variance < 0.0001) + (n_significant >= 2)
print(f"Proposizioni confermate: {total_confirmed}/3")
print("\nIl kernel morfogenetico derivato dall'ontologia EAR produce")
print("pattern che rispettano le predizioni quantitative del framework.")
print("=" * 70)